# Caso 08 — El Mirador, hotel: ¿lleno es lo mismo que rentable?
## Día 1 — Primer contacto con los datos

En este primer cuaderno exploramos los **6 ficheros CSV** exportados de los sistemas de El Mirador (`hoteles.csv`, `tipos_habitacion.csv`, `canales.csv`, `clientes.csv`, `reservas.csv` y `costes_operativos.csv`) **sin limpiar ni modificar nada todavía**.

### Objetivos del Día 1:
1. Comprobar dimensiones (`.shape`), primeras filas (`.head()`), tipos de datos (`dtypes`) y conteo de nulos (`.isna().sum()`) de las 6 tablas.
2. Entender el modelo en estrella con la **reserva en el centro** (`reservas.csv`, `107.417` filas en crudo) y sus dimensiones (`hoteles`, `tipos_habitacion`, `canales`, `clientes`) más la tabla mensual del ERP (`costes_operativos`).
3. Auditar las grafías sucias en categorías (`canal`, `estado`, `regimen`, `tipo` de hotel, `pais_origen`, `es_socio_fidelidad`), formatos de fecha mixtos e importes/porcentajes con símbolos (`€`, `%`).
4. Confirmar los avisos de **Dani Requena**:
   - **Duplicados por reintento de webhook:** `1.500` filas duplicadas por `reserva_id` (`107.417` filas totales frente a `105.917` reservas únicas).
   - **Los tres vacíos con significado de negocio en `reservas`:**
     1. `canal` vacío (`25.076` en crudo / `24.694` sin duplicados): **reserva directa** (0 % de comisión, las que más margen dejan).
     2. `fecha_cancelacion` vacía (`94.291` en crudo / `92.791` sin duplicados): **reserva NO cancelada**.
     3. `descuento_pct` vacío (`32.157` en crudo / `31.685` sin duplicados) y `fecha_salida` vacía (`17.456` en crudo / `17.455` sin duplicados): reserva **sin descuento (0 %)** y **estancia en curso / confirmada sin checkout cerrado**.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DIR = Path("data/raw")

tablas = {
    "hoteles": pd.read_csv(RAW_DIR / "hoteles.csv"),
    "tipos_habitacion": pd.read_csv(RAW_DIR / "tipos_habitacion.csv"),
    "canales": pd.read_csv(RAW_DIR / "canales.csv"),
    "costes_operativos": pd.read_csv(RAW_DIR / "costes_operativos.csv"),
    "clientes": pd.read_csv(RAW_DIR / "clientes.csv"),
    "reservas": pd.read_csv(RAW_DIR / "reservas.csv"),
}

resumen_tablas = pd.DataFrame([
    {
        "tabla": nombre,
        "filas": df.shape[0],
        "columnas": df.shape[1],
        "nulos_totales": int(df.isna().sum().sum()),
        "lista_columnas": ", ".join(df.columns),
    }
    for nombre, df in tablas.items()
])
display(resumen_tablas)


,tabla,filas,columnas,nulos_totales,lista_columnas
0,hoteles,5,6,0,"hotel_id, nombre, ciudad, tipo, categoria, hab..."
1,tipos_habitacion,7,4,1,"tipo_hab_id, nombre, capacidad, tarifa_base"
2,canales,6,4,0,"canal_id, nombre, tipo, comision_pct"
3,costes_operativos,60,5,0,"coste_id, hotel_id, mes, coste_fijo, coste_var..."
4,clientes,60000,5,1812,"cliente_id, fecha_nacimiento, pais_origen, es_..."
5,reservas,107417,15,168980,"reserva_id, hotel_id, cliente_id, tipo_hab_id,..."


### Análisis del mapa general de tablas
* El modelo gira alrededor de **`reservas`** (`107.417` filas y `15` columnas), que concentra casi todos los valores nulos (`168.980` celdas vacías repartidas en cuatro campos específicos).
* Las tablas maestras son pequeñas y manejables: **`hoteles`** (`5` hoteles del grupo), **`tipos_habitacion`** (`7` categorías de habitación) y **`canales`** (`6` canales de distribución).
* **`costes_operativos`** tiene `60` filas (`5` hoteles × `12` meses de 2024) sin nulos, exportadas del ERP de administración.
* **`clientes`** contiene `60.000` fichas únicas de huéspedes con `1.812` nulos únicamente en `pais_origen`.

A continuación inspeccionamos las tablas maestras y de costes para verificar sus tipos y formatos en crudo.


In [2]:
print("=== 1. HOTELES (5 hoteles del grupo) ===")
display(tablas["hoteles"])

print("\n=== 2. CANALES (6 canales canónicos y su comisión) ===")
display(tablas["canales"])

print("\n=== 3. TIPOS DE HABITACIÓN (7 categorías) ===")
display(tablas["tipos_habitacion"])

print("\n=== 4. COSTES OPERATIVOS (primeras 8 filas de 60: 5 hoteles x 12 meses) ===")
display(tablas["costes_operativos"].head(8))


=== 1. HOTELES (5 hoteles del grupo) ===


,hotel_id,nombre,ciudad,tipo,categoria,habitaciones
0,1,El Mirador Costa,MÁLAGA,Vacacional,4*,220
1,2,El Mirador Playa Sur,Cádiz,resort,4,165
2,3,El Mirador Centro,Madrid,ciudad,4*,165
3,4,El Mirador Ciudad,Sevilla,urbano,4,150
4,5,El Mirador Sierra,Granada,ciudad,3,110



=== 2. CANALES (6 canales canónicos y su comisión) ===


,canal_id,nombre,tipo,comision_pct
0,1,directo,directo,0 %
1,2,telefono,directo,0 %
2,3,booking,ota,18 %
3,4,expedia,ota,20 %
4,5,turoperador,Turoperador,25 %
5,6,corporate,Corporate,8 %



=== 3. TIPOS DE HABITACIÓN (7 categorías) ===


,tipo_hab_id,nombre,capacidad,tarifa_base
0,1,Individual,1,"65,00 €"
1,2,doble,2,"95,00 €"
2,3,Doble Superior,2,"120,00 €"
3,4,Junior Suite,2,NaN
4,5,Suite,3,"210,00 €"
5,6,fam,4,"140,00 €"
6,7,atico,3,"240,00 €"



=== 4. COSTES OPERATIVOS (primeras 8 filas de 60: 5 hoteles x 12 meses) ===


,coste_id,hotel_id,mes,coste_fijo,coste_variable_noche
0,1,1,2024-01-01,"113.840,24 €","26,00 €"
1,2,1,2024-02-01,"107.366,03 €","26,00 €"
2,3,1,2024-03-01,"116.403,13 €","26,00 €"
3,4,1,2024-04-01,"136.905,47 €","26,00 €"
4,5,1,01/05/2024,"122.598,93 €","26,00 €"
5,6,1,2024-06-01,"131.038,73 €","26,00 €"
6,7,1,01/07/2024,"129.194,44 €","26,00 €"
7,8,1,2024-08-01,"122.910,24 €","26,00 €"


### Análisis de las tablas de contexto y costes (`hoteles`, `canales`, `tipos_habitacion`, `costes_operativos`)
1. **`hoteles.csv`:**
   * Suma **810 habitaciones** entre los 5 hoteles (`220` en Costa, `165` en Playa Sur, `165` en Centro, `150` en Ciudad y `110` en Sierra).
   * La columna `tipo` mezcla grafías (`Vacacional`, `resort`, `ciudad`, `urbano`), y `categoria` mezcla números con y sin asterisco (`4*`, `4`, `3`). Además, mañana incorporaremos la disponibilidad anual oficial de negocio (`noches_disponibles`: `80.000` en Costa, `60.000` en Playa Sur y Centro, `55.000` en Ciudad y `40.000` en Sierra; `295.000` noches-habitación en total).
2. **`canales.csv`:**
   * Define los **6 canales canónicos** (`directo`, `telefono`, `booking`, `expedia`, `turoperador`, `corporate`).
   * `comision_pct` viene como texto con el símbolo `%` (`0 %`, `18 %`, `20 %`, `25 %`, `8 %`), por lo que habrá que limpiarlo y dividir entre 100 antes de multiplicar.
3. **`tipos_habitacion.csv`:**
   * `tarifa_base` viene como texto en formato moneda española (`65,00 €`) y presenta **1 nulo** en `Junior Suite` (`tipo_hab_id = 4`). Los nombres mezclan mayúsculas y abreviaturas (`doble`, `fam`, `atico`).
4. **`costes_operativos.csv`:**
   * Tiene 12 filas por hotel (`60` en total), pero la columna `mes` mezcla tres formatos de fecha (`2024-01-01`, `01/05/2024`, `1 Sep 2024`) y tanto `coste_fijo` como `coste_variable_noche` vienen como texto con punto de miles, coma decimal y símbolo `€` (`113.840,24 €`, `26,00 €`).


In [3]:
res = tablas["reservas"]

total_filas = len(res)
reservas_unicas = res["reserva_id"].nunique()
duplicados_id = res["reserva_id"].duplicated().sum()

print(f"Filas totales en reservas.csv : {total_filas:,}")
print(f"reserva_id únicos             : {reservas_unicas:,}")
print(f"Filas duplicadas por webhook  : {duplicados_id:,}")

res_dedup = res.drop_duplicates(subset=["reserva_id"])

comparativa_nulos = pd.DataFrame({
    "dtype_crudo": res.dtypes.astype(str),
    "nulos_crudo (107.417)": res.isna().sum(),
    "nulos_dedup (105.917)": res_dedup.isna().sum(),
    "no_nulos_dedup": res_dedup.notna().sum(),
})
display(comparativa_nulos)


Filas totales en reservas.csv : 107,417
reserva_id únicos             : 105,917
Filas duplicadas por webhook  : 1,500


,dtype_crudo,nulos_crudo (107.417),nulos_dedup (105.917),no_nulos_dedup
reserva_id,int64,0,0,105917
hotel_id,int64,0,0,105917
cliente_id,int64,0,0,105917
tipo_hab_id,int64,0,0,105917
canal,str,25076,24694,81223
fecha_reserva,str,0,0,105917
fecha_entrada,str,0,0,105917
fecha_salida,str,17456,17455,88462
noches,int64,0,0,105917
num_habitaciones,int64,0,0,105917


### Confirmación de los avisos de Dani Requena: duplicados y vacíos con significado
* **Duplicados confirmados:** De las `107.417` filas en `reservas.csv`, exactamente **`1.500` son duplicados** de `reserva_id` provocados por reintentos del webhook de la pasarela y sincronizaciones entre PMS y Channel Manager. Al deduplicar quedan **`105.917` reservas únicas**.
* **Los tres vacíos con significado de negocio (no son datos perdidos que deban borrarse):**
  1. **`canal` vacío (`25.076` en crudo / `24.694` tras deduplicar):** Son **reservas directas** que entraron por la web propia o recepción sin pasar por el Channel Manager. Son las más rentables (0 % de comisión).
  2. **`fecha_cancelacion` vacía (`94.291` en crudo / `92.791` tras deduplicar):** Significa que la reserva **no se canceló**. Esas **`92.791` reservas no canceladas** son la base sobre la que mañana aplicaremos el filtro de reserva consumida.
  3. **`descuento_pct` vacío (`32.157` en crudo / `31.685` tras deduplicar) y `fecha_salida` vacía (`17.456` en crudo / `17.455` tras deduplicar):** `descuento_pct` nulo indica **0 % de descuento** (tarifa sin rebaja promocional), y `fecha_salida` nula corresponde a reservas confirmadas o en curso cuyo checkout aún no se ha cerrado en el PMS.


In [4]:
print("=== GRAFÍAS DE 'canal' EN RESERVAS (25 variantes incl. NaN) ===")
print(res_dedup["canal"].value_counts(dropna=False).to_dict())

print("\n=== GRAFÍAS DE 'estado' EN RESERVAS (13 variantes) ===")
print(res_dedup["estado"].value_counts(dropna=False).to_dict())

print("\n=== GRAFÍAS DE 'regimen' EN RESERVAS (10 variantes) ===")
print(res_dedup["regimen"].value_counts(dropna=False).to_dict())

print("\n=== MUESTRA DE FORMATOS DE FECHA E IMPORTES EN RESERVAS ===")
display(res_dedup[["reserva_id", "fecha_reserva", "fecha_entrada", "fecha_salida", "tarifa_noche", "descuento_pct", "noches", "num_habitaciones"]].head(10))

print("\n=== GRAFÍAS EN CLIENTES ('pais_origen' y 'es_socio_fidelidad') ===")
print("pais_origen:", tablas["clientes"]["pais_origen"].value_counts(dropna=False).to_dict())
print("es_socio_fidelidad:", tablas["clientes"]["es_socio_fidelidad"].value_counts(dropna=False).to_dict())


=== GRAFÍAS DE 'canal' EN RESERVAS (25 variantes incl. NaN) ===
{nan: 24694, 'ota': 5589, 'Booking': 5527, 'Expedia': 5513, 'Booking.com': 5422, 'expedia': 5406, 'booking': 5401, 'BKG': 5319, 'EXP': 5309, 'turoperador': 3257, 'mayorista': 3241, 'grupo': 3215, 'TTOO': 3127, 'corporate': 2766, 'Corporate': 2741, 'empresa': 2701, 'corp': 2696, 'telefono': 2480, 'tel': 2436, 'Telefono': 2419, 'teléfono': 2334, 'web': 1102, 'directa': 1095, 'directo': 1074, 'Directo': 1053}

=== GRAFÍAS DE 'estado' EN RESERVAS (13 variantes) ===
{'checkout': 25119, 'completada': 25042, 'Completada': 25019, 'Confirmada': 4488, 'confirmada': 4434, 'ok': 4414, 'anulada': 4407, 'cancelada': 4367, 'cancel': 4352, 'noshow': 1096, 'No Show': 1080, 'no_show': 1068, 'no show': 1031}

=== GRAFÍAS DE 'regimen' EN RESERVAS (10 variantes) ===
{'AD': 24415, 'alojamiento y desayuno': 24335, 'SA': 8767, 'solo alojamiento': 8742, 'media pension': 7450, 'media pensión': 7340, 'MP': 7327, 'pension completa': 5858, 'pensión co

,reserva_id,fecha_reserva,fecha_entrada,fecha_salida,tarifa_noche,descuento_pct,noches,num_habitaciones
0,700177,04/04/2024,2024-05-09,2024-05-10,"64,12 €",3 %,1,1
1,753618,16/11/2023,2024-02-21,2024-02-24,"131,92 €",8 %,3,1
2,781100,13/03/2024,2024-03-26,2024-03-27,"77,94 €",NaN,1,1
3,790179,22/04/2024,28/07/2024,NaN,"97,44 €",3 %,1,1
4,717083,2024-05-02,2024-08-10,2024-08-11,"62,17 €",16 %,1,1
5,757528,2023-12-07,2024-02-20,2024-02-23,"113,52 €",NaN,3,1
6,770389,13/04/2024,08/07/2024,2024-07-09,"101,56 €",16 %,1,1
7,746223,14/01/2024,2024-01-24,26 Jan 2024,"127,85 €",NaN,2,1
8,769177,10 Feb 2024,26/02/2024,2024-03-01,"94,69 €",14 %,4,1
9,780321,2024-07-21,12 Sep 2024,16/09/2024,"94,06 €",3 %,4,1



=== GRAFÍAS EN CLIENTES ('pais_origen' y 'es_socio_fidelidad') ===
pais_origen: {'España': 9737, 'espana': 9697, 'Espana': 9591, 'ES': 9504, 'GB': 2704, 'Reino Unido': 2694, 'UK': 2673, 'Alemania': 2333, 'DE': 2303, nan: 1812, 'FR': 1782, 'Francia': 1771, 'Italia': 1142, 'IT': 1139, 'NL': 388, 'Holanda': 372, 'Paises Bajos': 358}
es_socio_fidelidad: {'false': 9355, 'No': 9333, '0': 9303, 'no': 9226, '1': 4614, 'true': 4569, 'SI': 4557, 'si': 4526, 'Sí': 4517}


### Auditoría de grafías sucias y formatos de texto
1. **`canal` en `reservas` (24 grafías de texto + `NaN`):** Mezcla mayúsculas, abreviaturas y sinónimos (`Booking`, `Booking.com`, `booking`, `BKG`, `ota`; `Expedia`, `expedia`, `EXP`; `turoperador`, `mayorista`, `grupo`, `TTOO`; `corporate`, `Corporate`, `corp`, `empresa`; `telefono`, `tel`, `Telefono`, `teléfono`; `web`, `directa`, `directo`, `Directo` + `NaN`). Todo debe converger mañana en los 6 canales canónicos de `canales.csv`.
2. **`estado` en `reservas` (13 grafías):** Variantes de cuatro estados reales: `completada` (`checkout`, `completada`, `Completada`), `confirmada` (`Confirmada`, `confirmada`, `ok`), `cancelada` (`anulada`, `cancelada`, `cancel`) y `no_show` (`noshow`, `No Show`, `no_show`, `no show`).
3. **`regimen` en `reservas` (10 grafías):** Cuatro regímenes (`SA`, `AD`, `MP`, `PC`) escritos como siglas y como texto con o sin tilde.
4. **Fechas e importes:** Las fechas mezclan ISO (`YYYY-MM-DD`), formato español (`DD/MM/YYYY`) y formato con mes abreviado en inglés (`12 Mar 2024`). `tarifa_noche` viene como texto con `€` y coma decimal, y `descuento_pct` con `%`.
5. **`clientes.csv`:** `pais_origen` tiene 16 grafías para 6 países (`ES`, `GB`, `DE`, `FR`, `IT`, `NL`) más `1.812` nulos reales, y `es_socio_fidelidad` mezcla booleanos, números y texto con/sin tilde (`false`, `No`, `0`, `no`, `1`, `true`, `SI`, `si`, `Sí`).

---

### Conclusión del Día 1
Hemos completado el mapa del modelo de datos de **El Mirador** sin tocar ni borrar una sola fila:
* Identificamos las **1.500 reservas duplicadas** (`107.417` $ightarrow$ `105.917` únicas).
* Validamos los **tres vacíos con significado de negocio** (`canal` nulo = directo sin comisión, `fecha_cancelacion` nula = `92.791` reservas no canceladas, `descuento_pct` nulo = 0 % descuento / `fecha_salida` nula = estancia sin checkout cerrado).
* Catalogamos todas las conversiones de tipos (fechas en 3 formatos, moneda con `€` y miles, porcentajes con `%`) y diccionarios de normalización que aplicaremos mañana en el **Día 2** para construir la definición exacta de **reserva consumida**.
